# Drinking Water Demand Forecasting with Recurrent Neural Networks - Part 4
## Probabilistic forecasting with Monte Carlo Dropout

**Probabilistic forecasting** should be preferred to "point forecasts" (e.g., predicting a single value or outcome) as it provides a measure of uncertainty of the predictions.

In this notebook, we will explore how to transform the Recurrent Neural Networks (RNNs) for water demand prediction into probabilistic forecasting models by using **dropout**. Dropout, known for its simplicity and effectiveness, is a form of regularization that randomly deactivates neurons during *training*. This technique prevents the model from becoming overly dependent on specific neurons, promoting a more generalized learning process. Building on this technique, **Monte Carlo Dropout**, where dropout is kept active during *infeerence*, allows the network to generate multiple forecasts by emulating ensembling.

The effectiveness of dropout for regularization or ensembling depends on fine-tuning two key hyperparameters: the *dropout rate* and the specific layers where dropout is applied. These hyperparameters influence the model's performance, balancing the need for regularization with the geeneration of informative prediction intervals.

In probabilistic forecasting, the goal is to achieve prediction intervals that are both accurate in **coverage** and **narrow** in width. This is usually checked on a validation dataset, where the aim is to minimize the intervals width while ensuring most actual values fall within these intervals. These aspects are evaluated using probabilistic metrics like **Prediction Interval Coverage Probability (PICP)** and **Prediction Interval Normalized Average Width (PINAW)**. PICP assesses how well the intervals cover the actual values, while PINAW evaluates the average width of these intervals, giving an indication of their precision.

In summary, dropout in RNNs for probabilistic forecasting offers a powerful combination of simplicity and effectiveness. It not only improves model robustness but also enables the creation of ensemble forecasts for a nuanced understanding of future possibilities. Key to its success is the careful adjustment of dropout rates and application areas, as well as a focused assessment of prediction intervals' coverage and width.

## Walkthrough


### Preliminary steps: load modules and data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import time
import os
from urllib.request import urlretrieve

from sklearn.preprocessing import StandardScaler,MinMaxScaler
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
# Check if CUDA is available, otherwise use CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Download the water demand dataset
year_1 = "https://surfdrive.surf.nl/files/index.php/s/hCXnLcb9zlYQHQd/download"
year_2 = "https://surfdrive.surf.nl/files/index.php/s/daXOMuxonOQL1oy/download"

data_folder = "data"
water_demand_file1 = os.path.join(data_folder, "water_demand_Y1.txt")
water_demand_file2 = os.path.join(data_folder, "water_demand_Y2.txt")

if not os.path.isfile(water_demand_file1):
    print("Downloading dataset...")
    os.makedirs("data", exist_ok=True)
    urlretrieve(year_1, water_demand_file1)
    urlretrieve(year_2, water_demand_file2)

In [ ]:
# load water demand data
df_year1 = pd.read_csv(water_demand_file1,header=None, sep=r"\s+")
df_year2 = pd.read_csv(water_demand_file2,header=None, sep=r"\s+")
df = pd.concat([df_year1, df_year2], axis = 0).reset_index(drop=True)
df.head()

### Create dataset and data loaders

In [5]:
def create_sequences(series,T=168,H=24):
    # This function creates a dataset of input/output sequences from a time series.
    # The input sequence is T steps long, from time t to time t+T (excluded).
    # The output sequence is H steps long, from time t+T to time t+T+H (excluded).
    X = []
    Y = []
    for t in range(len(series)-T-H):
        x = series[t:t+T]
        X.append(x)
        y = series[t+T:t+T+H]
        Y.append(y)
    X = np.array(X)
    Y = np.array(Y)
    return X,Y

def scale_sequences(X,scaler=None,scaler_type='standard'):
    # Uses a standard scaler to transform sequences. The scaler is created if no scaler is passed as argument.
    Xshape=X.shape
    if scaler:
        X = scaler.transform(X.reshape(-1,1)).reshape(Xshape)
        return X
    else:
        if scaler_type == 'standard':
            scaler = StandardScaler()
        elif scaler_type == 'minmax':
            scaler = MinMaxScaler()
        else:
            raise Exception("Type of scikit-learn scaler not supported. Choose 'standard' or 'minmax.")
        X = scaler.fit_transform(X.reshape(-1,1)).reshape(Xshape)
        return X, scaler

In [ ]:
T = 168     # number of time steps to use for prediction (168 hours = 1 week)
H = 24      # number of time steps to predict (1 day)
X, Y = create_sequences(df.values.reshape(-1),T=T,H=H)
print(X.shape)
print(Y.shape)

In [ ]:
# We keep track of indexes of train and validation.
X_tra, X_tst, Y_tra, Y_tst, ix_tra, ix_tst = train_test_split(
    X, Y, np.arange(X.shape[0]), test_size=0.30, shuffle=True, random_state=42)

# Split the existing test dataset into validation and test sets (50/50 split)
X_val, X_tst, Y_val, Y_tst, ix_val, ix_tst = train_test_split(
    X_tst, Y_tst, ix_tst, test_size=0.5, shuffle=True, random_state=42)


print(f"X_tra.shape: {X_tra.shape}, Y_tra.shape: {Y_tra.shape}")
print(f"X_val.shape: {X_val.shape}, Y_val.shape: {Y_val.shape}")
print(f"X_tst.shape: {X_tst.shape}, Y_tst.shape: {Y_tst.shape}")

In [8]:
# scale/normalize
X_tra, scaler = scale_sequences(X_tra, scaler_type='standard')
Y_tra = scale_sequences(Y_tra, scaler)
X_val = scale_sequences(X_val, scaler)
Y_val = scale_sequences(Y_val, scaler)
X_tst = scale_sequences(X_tst, scaler)
Y_tst = scale_sequences(Y_tst, scaler)

In [9]:
train_dataset = TensorDataset(torch.tensor(X_tra, dtype=torch.float32), torch.tensor(Y_tra, dtype=torch.float32))
val_dataset = TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(Y_val, dtype=torch.float32))
test_dataset = TensorDataset(torch.tensor(X_tst, dtype=torch.float32), torch.tensor(Y_tst, dtype=torch.float32 ))

batch_size = 256      # You can modify this based on your requirements

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

### GatedRNN class and training functions

In [10]:
class GatedRNN(nn.Module):
    def __init__(self, hidden_size, output_size, rnn_type='LSTM', dropout_rate = 0):
        super(GatedRNN, self).__init__()

        # Check the rnn_type and initialize the appropriate RNN layer (GRU or LSTM)
        if rnn_type == 'GRU':
            self.rnn = nn.GRU(input_size=1, hidden_size=hidden_size, batch_first=True)
        elif rnn_type == 'LSTM':
            self.rnn = nn.LSTM(input_size=1, hidden_size=hidden_size, batch_first=True)
        else:
            raise ValueError("Invalid RNN type. Choose 'GRU' or 'LSTM'")

        # Output layer
        self.dropout = nn.Dropout(p = dropout_rate)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # Reshape input to have feature dimension of 1
        x = x.unsqueeze(-1)   # Assuming input x has shape (batch, sequence)

        # RNN layer
        x, _ = self.rnn(x)   # We do not need the hidden states

        # Select the output of the last time step
        x = x[:, -1, :]

        # Output layer
        x = self.dropout(x)
        x = self.fc(x)

        return x

In [11]:
def evaluate_model(model, test_loader, criterion, device):
    model.eval()  # Set the model to evaluation mode
    test_loss = 0

    with torch.no_grad():  # No need to track gradients during evaluation
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            test_loss += loss.item()

    avg_test_loss = test_loss / len(test_loader)
    return avg_test_loss

def train_and_validate(model, train_loader, val_loader, criterion, optimizer, num_epochs, device, save_path, echo_iter = 20):
    best_val_loss = float("inf")  # Track the best validation loss
    train_losses = []
    val_losses = []

    start_time = time.time()  # Start training time

    for epoch in range(num_epochs):
        # Training Phase
        model.train()
        total_train_loss = 0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # Validation Phase
        avg_val_loss = evaluate_model(model, val_loader, criterion, device)
        val_losses.append(avg_val_loss)

        # Save Best Model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), save_path)

        if echo_iter > 0:
            if (epoch + 1) % echo_iter == 0:
                print(f'Epoch {epoch+1}/{num_epochs}', f'Train Loss: {avg_train_loss:.4f}, '
                      f'Validation Loss: {avg_val_loss:.4f}', f'Best Validation Loss: {best_val_loss:.4f}')

    train_time = time.time() - start_time
    if echo_iter > 0:
        print("Training complete.")
    return train_losses, val_losses, best_val_loss, train_time

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

### Train and test GatedRNN with Dropout

In [12]:
!mkdir models

In [13]:
# train GatedRNN model with dropout
model_24hours_dropout = GatedRNN(128,24,'LSTM',dropout_rate = 0.2).to(device)
save_path = f'./models/model_24hours_dropout.pth'

In [ ]:
lr = 0.001
criterion = nn.MSELoss()
optimizer = optim.AdamW(model_24hours_dropout.parameters(), lr=lr)
num_epochs = 1000
train_losses, val_losses, _, elapsed_time = train_and_validate(model_24hours_dropout, train_loader, val_loader,
                                                               criterion, optimizer, num_epochs, device, save_path, echo_iter = 100)

In [ ]:
# load best weights
model_24hours_dropout.load_state_dict(torch.load(save_path, map_location=torch.device(device)))

criterion = nn.MSELoss()  # Or another appropriate loss function
avg_test_loss = evaluate_model(model_24hours_dropout, test_loader, criterion, device)
print(f"GatedRNN --> num. trainable parameters:{count_parameters(model_24hours_dropout):8d} | Test loss: {avg_test_loss:.4f}")

###  Compute and visualize Monte Carlo dropout prediction intervals

#### Define functions for computation of prediction intervals and probabilistic metrics

In [16]:
def generate_dropout_predictions(model, input_data, num_samples=30):
    model.train()  # Enable dropout
    with torch.no_grad():  # Disable gradient calculations
        predictions = [model(input_data) for _ in range(num_samples)]
    return torch.stack(predictions, dim=0)

In [17]:
def calculate_prediction_intervals(predictions, percentile=95):
    """
    Calculate prediction intervals from multiple predictions.

    Parameters:
    - predictions: Array of predictions.
    - percentile: The confidence level for the interval.

    Returns:
    - lower_bound: Lower bounds of prediction intervals.
    - upper_bound: Upper bounds of prediction intervals.
    """
    lower_percentile = (100 - percentile) / 2
    upper_percentile = 100 - lower_percentile

    lower_bound = torch.quantile(predictions, lower_percentile / 100, dim=0)
    upper_bound = torch.quantile(predictions, upper_percentile / 100, dim=0)

    return lower_bound, upper_bound


def calculate_picp(y_true, lower_bound, upper_bound):
    """
    Calculate the Prediction Interval Coverage Probability (PICP).

    Parameters:
    - y_true: Actual values.
    - lower_bound: Lower bound of the prediction intervals.
    - upper_bound: Upper bound of the prediction intervals.

    Returns:
    - PICP value.
    """
    coverage = torch.where((y_true >= lower_bound) & (y_true <= upper_bound), 1, 0)
    picp = torch.mean(coverage, dtype=float)
    return picp * 100

def calculate_pinaw(y_true, lower_bound, upper_bound, epsilon=1e-8):
    """
    Calculate the Prediction Interval Normalized Average Width (PINAW)
    normalized by the range of y_true.

    Parameters:
    - y_true: Actual values (Tensor).
    - lower_bound: Lower bound of the prediction intervals (Tensor).
    - upper_bound: Upper bound of the prediction intervals (Tensor).
    - epsilon: Small value to avoid division by zero (default: 1e-8).

    Returns:
    - PINAW value (float).
    """
    # Calculate interval width
    interval_width = upper_bound - lower_bound  # Shape: [N]

    # Calculate the range of y_true
    data_range = torch.max(y_true) - torch.min(y_true)

    # Add small epsilon for stability in case the range is 0
    normalized_width = interval_width / (data_range + epsilon)  # Normalized width

    # Mean of normalized widths
    pinaw = torch.mean(normalized_width)
    return pinaw

In [18]:
def compute_picp_pinaw_loader(model, loader, percentile = 95, num_samples=30):
    all_picps  = []
    all_pinaws = []

    for inputs, targets in loader:
        inputs, targets = inputs.to(device), targets.to(device)
        predictions = generate_dropout_predictions(model, inputs, num_samples=num_samples)

        # Compute upper and lower bound of prediction intervals
        lb, ub = calculate_prediction_intervals(predictions, percentile)

        # Compute metrics
        picp = calculate_picp(targets, lb, ub)
        pinaw = calculate_pinaw(targets, lb, ub)

        all_picps.append(picp.cpu().numpy())
        all_pinaws.append(pinaw.cpu().numpy())

    return np.array(all_picps).mean(), np.array(all_pinaws).mean()

#### Compute average metrics for all datasets

In [ ]:
# Return average metrics (95% percentile) for training, validation and test
tra_picp, tra_pinaw = compute_picp_pinaw_loader(model_24hours_dropout, train_loader, percentile=95, num_samples=100)
val_picp, val_pinaw = compute_picp_pinaw_loader(model_24hours_dropout, val_loader, percentile=95, num_samples=100)
tst_picp, tst_pinaw = compute_picp_pinaw_loader(model_24hours_dropout, test_loader, percentile=95, num_samples=100)

print(f"Training dataset:\t PICP={tra_picp:.2f}% pinaw={tra_pinaw:.4f}")
print(f"Validation dataset:\t PICP={val_picp:.2f}% pinaw={val_pinaw:.4f}")
print(f"Test dataset:\t\t PICP={tst_picp:.2f}% pinaw={tst_pinaw:.4f}")

#### Visualize individual prediction intervals for the test dataset

In [ ]:
# Create subplots
f, axes = plt.subplots(2, 4, figsize=(20, 12))

# Create a new test loader for visualization purposes in the loop below
test_loader_batch1 = DataLoader(test_dataset, batch_size=1, shuffle=True)

for ix, ax in enumerate(axes.reshape(-1)):
    inputs, targets = next(iter(test_loader_batch1))

    # Predict
    predictions = generate_dropout_predictions(model_24hours_dropout, inputs.to(device), num_samples=100)

    # Scale back to actual values
    targets = scaler.inverse_transform(targets)
    predictions = scaler.inverse_transform(predictions.cpu().detach().squeeze().numpy())

    # Plotting ensemble predictions
    for i in range(predictions.shape[0]):
        if i == 0:
            ax.plot(np.arange(T, T+H), predictions[i,:], linestyle='-', color='lightgray', label='ensemble prediction')
        else:
            ax.plot(np.arange(T, T+H), predictions[i,:], linestyle='-', color='lightgray')

    # Plot lower and upper bounds and compute metrics
    lb, ub = calculate_prediction_intervals(torch.tensor(predictions), 95)
    ax.plot(np.arange(T, T+H), lb, linestyle=':', color='blue', label='lower/upper bound')
    ax.plot(np.arange(T, T+H), ub, linestyle=':', color='blue')

    # Plot actual targets
    ax.plot(np.arange(T, T+H), targets.squeeze(), '--g', label='target')
    for t in range(T, T+H):
        target_value = targets.squeeze()[t-T]
        if target_value < lb[t-T] or target_value > ub[t-T]:
            # Target is outside the bounds, use a different color (e.g., red)
            ax.plot(t, target_value, 'rx')  # 'rx' for red 'x'
        else:
            # Target is inside the bounds, use the standard color (e.g., green)
            ax.plot(t, target_value, 'gx')  # 'gx' for green 'x'
    # Plot dummy lines for legend
    ax.plot([], [], 'gx', label='target (inside)')
    ax.plot([], [], 'rx', label='target (outside)')

    # Compute metrics
    picp = calculate_picp(torch.tensor(targets), lb, ub)
    pinaw = calculate_pinaw(torch.tensor(targets), lb, ub)

    # Finalize plot
    ax.set_title(f'Test Example #{ix+1:02d} | PICP={picp:.2f}% PINAW={pinaw:.4f}')
    ax.set_xlabel('Time Steps')
    ax.set_ylabel('Value')
    if ix == 0:
        ax.legend()

f.tight_layout()

## Assignments

1. **Effect of dropout rate on training process and model generalization**
  - What is the effect of dropout regularization on the training process and the generalization performances of your model?
  - Try a couple of values, as well as removing dropout by setting it to 0. Discuss the results.

2. **Optimize dropout rate for probabilistic forecasting**
  - Try several dropout rates and check how the PICP and PINAW metrics change with it.
  - Ideally, you want to select the dropout rate yielding a PICP >= expected coverage (e.g., 95%) with the smallest PINAW (i.e., narrowest).
  - What is the optimal value you find?
  - Compute new test predictions by averaging the ensembles obtained with the optimal dropout. Compute the test MSE between target data and these predictions. How does it compare against the one obtained in the traditional way (e.g., point predictions).
  